# **Model Training (BERT)**

In [3]:
import os
import json
import argparse
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from tqdm import tqdm


# ─────────────────────────────────────────────────────────────
# 1.  ARGUMENT PARSING
# ─────────────────────────────────────────────────────────────

def parse_args(args=None):
    p = argparse.ArgumentParser(description="Fine-tune BERT for text classification")
    p.add_argument("--data_path",   type=str,   default="bert_ready_dataset.csv",
                   help="Path to the cleaned CSV (must contain 'clean_text' and 'target' columns)")
    p.add_argument("--output_dir",  type=str,   default="./bert_output",
                   help="Directory to save the model, tokenizer, and results")
    p.add_argument("--model_name",  type=str,   default="bert-base-uncased")
    p.add_argument("--max_len",     type=int,   default=256,
                   help="Maximum token length (512 max for BERT, 256 balances speed/coverage)")
    p.add_argument("--batch_size",  type=int,   default=16)
    p.add_argument("--epochs",      type=int,   default=4)
    p.add_argument("--lr",          type=float, default=2e-5,   help="Learning rate")
    p.add_argument("--weight_decay",type=float, default=0.01)
    p.add_argument("--warmup_ratio",type=float, default=0.1,
                   help="Fraction of total steps used for LR warm-up")
    p.add_argument("--val_split",   type=float, default=0.1,    help="Validation set fraction")
    p.add_argument("--test_split",  type=float, default=0.1,    help="Test set fraction")
    p.add_argument("--seed",        type=int,   default=42)
    p.add_argument("--fp16",        action="store_true",
                   help="Use mixed-precision training (requires CUDA)")
    p.add_argument("--dropout",     type=float, default=0.3,
                   help="Classifier dropout (hidden → logits)")
    return p.parse_args(args=args)


# ─────────────────────────────────────────────────────────────
# 2.  REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────────────────────
# 3.  DATASET
# ─────────────────────────────────────────────────────────────

class TextClassificationDataset(Dataset):
    """
    Wraps a list of texts and integer labels.
    Tokenization is done here (one sample at a time) to avoid loading
    the full (N × 512) tensor array into RAM when max_len is large.
    """

    def __init__(self, texts: list, labels: list, tokenizer, max_len: int):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length      = self.max_len,
            padding         = "max_length",
            truncation      = True,
            return_attention_mask   = True,
            return_token_type_ids   = True,
            return_tensors  = "pt",
        )
        return {
            "input_ids"      : encoding["input_ids"].squeeze(0),        # (max_len,)
            "attention_mask" : encoding["attention_mask"].squeeze(0),
            "token_type_ids" : encoding["token_type_ids"].squeeze(0),
            "label"          : torch.tensor(self.labels[idx], dtype=torch.long),
        }


# ─────────────────────────────────────────────────────────────
# 4.  TRAINING / EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, scheduler, device, scaler=None):
    model.train()
    total_loss  = 0.0
    all_preds   = []
    all_labels  = []

    for batch in tqdm(loader, desc="  Training", leave=False):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()

        if scaler:                          # fp16 path
            from torch.cuda.amp import autocast
            with autocast():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    labels=labels,
                )
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:                               # fp32 path
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels,
            )
            loss = outputs.loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def evaluate(model, loader, device, split_name="Validation"):
    model.eval()
    total_loss  = 0.0
    all_preds   = []
    all_labels  = []

    for batch in tqdm(loader, desc=f"  {split_name}", leave=False):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels         = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels,
        )

        total_loss += outputs.loss.item()
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="weighted")
    return avg_loss, acc, f1, all_preds, all_labels


# ─────────────────────────────────────────────────────────────
# 5.  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    # In a Colab environment, argparse might pick up arguments from the kernel.
    # Passing an empty list to parse_args() ensures only default arguments are used.
    args = parse_args(args=[])
    set_seed(args.seed)

    # ── Device ──────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*60}")
    print(f"  Device : {device}")
    if device.type == "cuda":
        print(f"  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"{'='*60}\n")

    # ── Output directory ────────────────────────
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True);

    # ── 5.1  Load data ───────────────────────────
    print("STEP 1 – Loading dataset")
    df = pd.read_csv(args.data_path)
    print(f"  Rows           : {len(df)}")
    print(f"  Columns        : {df.columns.tolist()}")
    assert "clean_text" in df.columns, "'clean_text' column not found!"
    assert "target"     in df.columns, "'target' column not found!"

    # Drop any remaining NaN texts
    df = df.dropna(subset=["clean_text"]).reset_index(drop=True)

    texts  = df["clean_text"].tolist()
    labels = df["target"].tolist()
    num_labels = len(set(labels))
    print(f"  Classes        : {sorted(set(labels))}  ({num_labels} total)")
    print(f"  Distribution   :\n{df['target'].value_counts().sort_index().to_string()}")

    # ── 5.2  Train / Val / Test split ──────────────────────
    print("\nSTEP 2 – Splitting dataset")
    test_size = args.test_split
    val_size  = args.val_split / (1.0 - test_size)   # val fraction of (train+val)

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        texts, labels, test_size=test_size, random_state=args.seed, stratify=labels
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val,
        test_size=val_size, random_state=args.seed, stratify=y_train_val
    )
    print(f"  Train : {len(X_train)}  |  Val : {len(X_val)}  |  Test : {len(X_test)}")

    # ── 5.3  Tokenizer ─────────────────────────
    print(f"\nSTEP 3 – Loading tokenizer ({args.model_name})")
    tokenizer = BertTokenizer.from_pretrained(args.model_name)

    # ── 5.4  DataLoaders ───────────────────────
    print("\nSTEP 4 – Creating DataLoaders")
    train_dataset = TextClassificationDataset(X_train, y_train, tokenizer, args.max_len)
    val_dataset   = TextClassificationDataset(X_val,   y_val,   tokenizer, args.max_len)
    test_dataset  = TextClassificationDataset(X_test,  y_test,  tokenizer, args.max_len)

    num_workers = min(4, os.cpu_count() or 1)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                              shuffle=True,  num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=args.batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=args.batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)
    print(f"  Batches — Train: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}")

    # ── 5.5  Model ─────────────────────
    print(f"\nSTEP 5 – Loading model ({args.model_name})")
    model = BertForSequenceClassification.from_pretrained(
        args.model_name,
        num_labels              = num_labels,
        hidden_dropout_prob     = args.dropout,
        attention_probs_dropout_prob = args.dropout,
        ignore_mismatched_sizes = True,
    )
    model = model.to(device)
    total_params = sum(p.numel() for p in model.parameters())
    trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params    : {total_params:,}")
    print(f"  Trainable params: {trainable:,}")

    # ── 5.6  Optimizer & Scheduler ─────────────────────
    # Layer-wise learning rate decay: encoder layers use a smaller LR
    no_decay  = ["bias", "LayerNorm.weight"]
    optimizer_grouped = [
        {
            "params": [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay)],
            "weight_decay": args.weight_decay,
        },
        {
            "params": [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    optimizer = AdamW(optimizer_grouped, lr=args.lr, eps=1e-8)

    total_steps  = len(train_loader) * args.epochs
    warmup_steps = int(total_steps * args.warmup_ratio)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    print(f"\n  Total training steps : {total_steps}")
    print(f"  Warmup steps         : {warmup_steps}")

    # Mixed precision scaler (fp16 only on CUDA)
    scaler = None
    if args.fp16 and device.type == "cuda":
        from torch.cuda.amp import GradScaler
        scaler = GradScaler()
        print("  Mixed-precision (fp16) enabled")

    # ── 5.7  Training loop ──────────────────────────
    print(f"\n{'='*60}")
    print(f"  TRAINING  ({args.epochs} epochs)")
    print(f"{'='*60}")

    history = {"train_loss": [], "train_acc": [],
               "val_loss":   [], "val_acc":   [], "val_f1": []}
    best_val_f1    = 0.0
    best_model_dir = output_dir / "best_model"

    for epoch in range(1, args.epochs + 1):
        print(f"\nEpoch {epoch}/{args.epochs}")

        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, device, scaler
        )
        vl_loss, vl_acc, vl_f1, _, _ = evaluate(model, val_loader, device, "Validation")

        history["train_loss"].append(round(tr_loss, 4))
        history["train_acc" ].append(round(tr_acc,  4))
        history["val_loss"  ].append(round(vl_loss, 4))
        history["val_acc"   ].append(round(vl_acc,  4))
        history["val_f1"    ].append(round(vl_f1,   4))

        print(f"  Train  — loss: {tr_loss:.4f}  acc: {tr_acc:.4f}")
        print(f"  Val    — loss: {vl_loss:.4f}  acc: {vl_acc:.4f}  f1(w): {vl_f1:.4f}")

        # Save best checkpoint
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            model.save_pretrained(str(best_model_dir))
            tokenizer.save_pretrained(str(best_model_dir))
            print(f"  ✓ Best model saved  (val_f1 = {best_val_f1:.4f})")

    # ── 5.8  Test evaluation (best checkpoint) ───────────────────
    print(f"\n{'='*60}")
    print("  TEST EVALUATION  (best checkpoint)")
    print(f"{'='*60}")

    best_model = BertForSequenceClassification.from_pretrained(str(best_model_dir))
    best_model = best_model.to(device)

    ts_loss, ts_acc, ts_f1, ts_preds, ts_labels = evaluate(
        best_model, test_loader, device, "Test"
    )
    print(f"\n  Test loss      : {ts_loss:.4f}")
    print(f"  Test accuracy  : {ts_acc:.4f}")
    print(f"  Test F1 (w)    : {ts_f1:.4f}")

    label_names = [str(c) for c in sorted(set(labels))]
    print("\n  Classification Report:\n")
    print(classification_report(ts_labels, ts_preds, target_names=label_names))

    print("\n  Confusion Matrix:")
    cm = confusion_matrix(ts_labels, ts_preds)
    print(cm)

    # ── 5.9  Save artefacts ─────────────────────
    # Training history
    history_path = output_dir / "training_history.json"
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)
    print(f"\n  Saved training history → {history_path}")

    # Final test results
    results = {
        "test_loss"    : round(ts_loss, 4),
        "test_accuracy": round(ts_acc,  4),
        "test_f1_weighted": round(ts_f1, 4),
        "best_val_f1"  : round(best_val_f1, 4),
        "epochs"       : args.epochs,
        "model_name"   : args.model_name,
        "max_len"      : args.max_len,
        "batch_size"   : args.batch_size,
        "learning_rate": args.lr,
        "num_labels"   : num_labels,
    }
    results_path = output_dir / "test_results.json"
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  Saved test results      → {results_path}")

    # Save final model (last epoch) as well
    final_model_dir = output_dir / "final_model"
    model.save_pretrained(str(final_model_dir))
    tokenizer.save_pretrained(str(final_model_dir))
    print(f"  Saved final model       → {final_model_dir}")

    print(f"\n{'='*60}")
    print("  Training complete ✓")
    print(f"{'='*60}\n")


# ─────────────────────────────────────────────────────────────
# 6.  INFERENCE HELPER
# ─────────────────────────────────────────────────────────────

def predict(texts: list, model_dir: str, max_len: int = 256, batch_size: int = 32):
    """
    Load the saved model and return predicted class indices for a list of texts.

    Parameters
    ----------
    texts      : list of str — raw (or pre-cleaned) text samples
    model_dir  : path to the saved model directory (bert_output/best_model)
    max_len    : should match the value used at training time
    batch_size : inference batch size

    Returns
    -------
    preds      : list of int — predicted class for each input text
    probs      : list of list[float] — softmax probabilities for each class
    """
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = BertTokenizer.from_pretrained(model_dir)
    model     = BertForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()

    all_preds = []
    all_probs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        encoding = tokenizer(
            batch_texts,
            max_length   = max_len,
            padding      = "max_length",
            truncation   = True,
            return_attention_mask   = True,
            return_token_type_ids   = True,
            return_tensors = "pt",
        )
        with torch.no_grad():
            outputs = model(
                input_ids      = encoding["input_ids"].to(device),
                attention_mask = encoding["attention_mask"].to(device),
                token_type_ids = encoding["token_type_ids"].to(device),
            )
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=-1).tolist()
        all_preds.extend(preds)
        all_probs.extend(probs.tolist())

    return all_preds, all_probs


# ──────────────────────────────────────────────────

if __name__ == "__main__":
    main()


  Device : cuda
  GPU    : Tesla T4

STEP 1 – Loading dataset
  Rows           : 4697
  Columns        : ['clean_text', 'title', 'text', 'target']
  Classes        : [0, 1, 2, 3, 4]  (5 total)
  Distribution   :
target
0    870
1    974
2    883
3    997
4    973

STEP 2 – Splitting dataset
  Train : 3757  |  Val : 470  |  Test : 470

STEP 3 – Loading tokenizer (bert-base-uncased)

STEP 4 – Creating DataLoaders
  Batches — Train: 235  Val: 30  Test: 30

STEP 5 – Loading model (bert-base-uncased)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

  ✓ Best model saved  (val_f1 = 0.7853)

  TEST EVALUATION  (best checkpoint)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


  Test loss      : 0.7383
  Test accuracy  : 0.7447
  Test F1 (w)    : 0.7465

  Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.76      0.80        87
           1       0.72      0.72      0.72        98
           2       0.75      0.72      0.73        88
           3       0.64      0.75      0.69       100
           4       0.81      0.77      0.79        97

    accuracy                           0.74       470
   macro avg       0.75      0.74      0.75       470
weighted avg       0.75      0.74      0.75       470


  Confusion Matrix:
[[66  4  5  8  4]
 [ 3 71  5 16  3]
 [ 4  6 63  9  6]
 [ 2 14  4 75  5]
 [ 2  4  7  9 75]]

  Saved training history → bert_output/training_history.json
  Saved test results      → bert_output/test_results.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Saved final model       → bert_output/final_model

  Training complete ✓

